# Extract Events

Parse document text into traceable punch events and page-level diagnostics.

**Requires:** document extraction manifests and JSON files.  
**Produces:** `events.csv`, `pages.csv`, and an extraction report.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("extract_events")
display(nb.pipeline_overview(ctx, "extract_events"))


## Controls


In [ ]:
VERBOSE = True
MAX_PATTERN_EXAMPLES = int(step_cfg.get("max_pattern_examples", 12))
MAX_UNMATCHED_EXAMPLES = int(step_cfg.get("max_unmatched_examples_per_file", 5))

{
    "max_pattern_examples": MAX_PATTERN_EXAMPLES,
    "max_unmatched_examples_per_file": MAX_UNMATCHED_EXAMPLES,
}


## Inputs


In [ ]:
display(nb.artifact_table({
    "included document index": paths.documents_included_index,
    "document report": paths.documents_report,
}))
display(nb.file_table(paths.documents_dir, "*.text_extraction.csv"))


## Build Options


In [ ]:
from core.events.extraction.options import ExtractEventsFromTextOptions

options = ExtractEventsFromTextOptions(
    input_dir=str(paths.documents_dir),
    output_dir=str(paths.events_dir),
    report_json=str(paths.events_report),
    max_pattern_examples=MAX_PATTERN_EXAMPLES,
    max_unmatched_examples_per_file=MAX_UNMATCHED_EXAMPLES,
    verbose=VERBOSE,
)
options


## Run Event Extraction


In [ ]:
from core.drive.logging_utils import setup_logging
from core.events.extraction.service import run_from_options

setup_logging(VERBOSE)
event_report = run_from_options(options)
display(nb.report_summary(event_report))


## Inspect Results


In [ ]:
display(nb.artifact_table({
    "events": paths.events_csv,
    "pages": paths.pages_csv,
    "event report": paths.events_report,
}))
display(nb.preview_csv(paths.events_csv))
display(nb.preview_csv(paths.pages_csv))
